# UPENN-GBM similar-case gallery — Colab build (milestone 1)

This notebook builds the **real 3D retrieval gallery** in the cloud, so the raw MRI scans never touch your laptop.

1. List the individual files in the GitHub mirror `data-nih/tcia` (release `upenn-gbm`)
2. Download the complete baseline cases (T1, T1-Gd, T2, FLAIR + tumour mask), resumably, onto the Colab VM
3. Validate every case: files open, all five share one geometry, mask labels are as expected
4. Check and record the `sub-NNN` → `UPENN-GBM-NNNNN_11` metadata join
5. Split by patient (gallery 70% / validation 15% / test 15%), extract interpretable features, fit scaling on the gallery only
6. Export a small **gallery bundle** (features, display metadata, PNG previews, experiment manifest, reports) to Google Drive for Streamlit

**Runtime:** CPU is enough (no GPU needed). Expect roughly 5–10 min of download and 10–20 min of feature extraction.
**Storage:** about 1.6 GB of raw files on the Colab VM disk (`/content`), which is deleted when the runtime ends. Only the bundle (a few MB) goes to Drive.

Data: UPENN-GBM, The Cancer Imaging Archive, DOI [10.7937/TCIA.709X-DN49](https://doi.org/10.7937/TCIA.709X-DN49), CC BY 4.0. Cite Bakas et al. (2022), *Scientific Data*, in anything you publish. This is a research prototype, not a diagnostic tool.

## 0. Settings
Pick **one** way to get the code:
* **A (recommended while the branch is local-only):** on your laptop run
  `git archive --format=zip -o retrieval_code.zip feature/upenn-gbm-retrieval` in the repo, upload `retrieval_code.zip` to `MyDrive/upenn_gbm_retrieval/`, and leave `CODE_ZIP` as is.
* **B (after *you* have pushed the branch to your fork):** set `REPO_URL`. The notebook never pushes anything.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/upenn_gbm_retrieval"   # bundle + reports go here (small)
CODE_ZIP   = f"{DRIVE_ROOT}/retrieval_code.zip"              # option A
REPO_URL   = ""                                                # option B, e.g. https://github.com/<you>/brain-tumor-detection.git
BRANCH     = "feature/upenn-gbm-retrieval"

RAW_DIR    = "/content/upenn_raw"   # raw scans live on the Colab VM only (not Drive, not your laptop)
# Set True to cache raw files in Drive so downloads resume across Colab sessions (~1.6 GB of Drive space).
# If you use Google Drive for desktop, exclude this folder from sync or it will reach your laptop.
KEEP_RAW_ON_DRIVE = False
SEED = 42
TOP_K = 5

## 1. Mount Drive and fetch the code

In [ ]:
import os, sys, subprocess, zipfile, pathlib
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
if KEEP_RAW_ON_DRIVE:
    RAW_DIR = f"{DRIVE_ROOT}/raw_cache"
os.makedirs(RAW_DIR, exist_ok=True)

CODE_DIR = "/content/code"
if os.path.exists(CODE_DIR):
    subprocess.run(["rm", "-rf", CODE_DIR], check=True)
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CODE_DIR], check=True)
elif os.path.exists(CODE_ZIP):
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
else:
    raise FileNotFoundError(f"Upload retrieval_code.zip to {DRIVE_ROOT} or set REPO_URL.")
sys.path.insert(0, CODE_DIR)
assert os.path.exists(f"{CODE_DIR}/retrieval/experiment.py"), "retrieval/ package not found in the code source"
print("code ready:", sorted(os.listdir(CODE_DIR))[:12])

In [ ]:
# Colab already ships numpy, pandas, scipy, scikit-learn and Pillow.
!pip -q install nibabel==5.3.2

## 2. Inventory the mirror
One GitHub API request lists every file. Baseline files are `sub-NNN_<T1w|ce-gd_T1w|T2w|FLAIR|seg>.nii.gz`;
follow-up files (`sub-NNN_2_*`) are counted but not used. On 2026-11-15 the listing had **4,424 files, 630 subjects
and 147 complete baseline cases (~1.60 GB)**. A lower total means the listing was truncated.

In [ ]:
import pandas as pd, json
from retrieval.upenn import fetch_release_assets, build_inventory, DOWNLOAD_BASE, download_file, read_participants

assets = fetch_release_assets()
inventory = build_inventory(assets)
complete = inventory[inventory["complete"]]
print(f"assets listed: {len(assets)}  subjects: {len(inventory)}  complete baseline cases: {len(complete)}  "
      f"size: {complete['total_bytes'].sum()/1e9:.2f} GB")
print("missing roles among incomplete subjects:")
print(inventory.loc[~inventory["complete"], "missing_roles"].value_counts().head())
inventory.to_csv(f"{DRIVE_ROOT}/inventory.csv", index=False)

for name in ("participants.tsv", "qc.tsv"):
    download_file(f"{DOWNLOAD_BASE}/{name}", pathlib.Path(RAW_DIR) / name)
participants = read_participants(pathlib.Path(RAW_DIR) / "participants.tsv")
print("participants rows:", len(participants))

## 3. Download (resumable)
Each file is written to `<name>.part` and resumed with HTTP Range requests. It is only renamed once its size matches the
listing. **If the cell is interrupted, run it again**: finished files are skipped and partial files resume.

In [ ]:
from retrieval.upenn import download_inventory
report = download_inventory(inventory, RAW_DIR, max_workers=6,
                            progress=lambda done, total: print(f"  {done}/{total} files"))
print(report["status"].value_counts())
failed = report[report["status"] == "failed"]
if len(failed):
    print("FAILED downloads (re-run this cell to retry):")
    print(failed.to_string(index=False))
report.to_csv(f"{DRIVE_ROOT}/download_report.csv", index=False)

## 4–6. Validate, join metadata, split by patient, extract features, export the bundle
`build_upenn_gallery` does the following, in order:
* opens all five files of every case and checks they share one shape, voxel spacing and orientation, and that the mask labels are in {0,1,2,3}, non-empty and not binary;
* checks the numeric ID mapping independently (below) and only exposes metadata for verified joins;
* splits **by patient** with a fixed seed, then extracts the same features for every case;
* fits the z-score scaler on the **gallery only** and reuses it for validation/test queries;
* writes previews for gallery and test cases, rankings for validation/test queries, and every report.

In [ ]:
from retrieval.experiment import build_upenn_gallery
BUNDLE_DIR = "/content/gallery_bundle"
subprocess.run(["rm", "-rf", BUNDLE_DIR], check=True)
manifest = build_upenn_gallery(inventory, participants, RAW_DIR, BUNDLE_DIR, download_report=report,
                               seed=SEED, top_k=TOP_K, progress=print)
print(json.dumps(manifest["counts"], indent=2))

### Exclusions and validation
Every subject that did not make it into the experiment is listed with the stage and the reason.

In [ ]:
exclusions = pd.read_csv(f"{BUNDLE_DIR}/reports/exclusions.csv")
print(exclusions.groupby(["stage"]).size().rename("subjects"))
display(exclusions[exclusions["stage"] != "inventory"])
validation = pd.read_csv(f"{BUNDLE_DIR}/reports/validation_report.csv")
print("shapes:", validation["shape"].value_counts().to_dict())
print("spacings:", validation["spacing"].value_counts().to_dict())
print("label sets:", validation["labels"].value_counts().to_dict())

### Mask label meaning
The mirror stores labels **1, 2, 3**, not the BraTS convention 1, 2, 4. We assume 1 = necrotic core, 2 = oedema,
3 = enhancing tumour, and **test that assumption** on intensities: enhancing tumour should be the brightest
region on post-contrast T1, and necrosis should be dark on T1-Gd.

In [ ]:
sem = manifest["label_semantics_check"]
display(pd.DataFrame(sem).T)
ok = (sem["label3"]["mean_t1gd_z"] > sem["label2"]["mean_t1gd_z"] > sem["label1"]["mean_t1gd_z"])
print("Label semantics consistent with 1=NCR, 2=ED, 3=ET:", ok)
assert ok, "Intensity pattern does not match the assumed labels. Stop and re-check the label meaning."

### Metadata join check
`sub-002 → UPENN-GBM-00002_11` is a **candidate** numeric mapping. We test it independently: the mirror marks which
subjects have follow-up scans (`sub-NNN_2_*`), and `participants.tsv` marks follow-up rows (`_21`). If the numbering
is the same, every mirror follow-up subject must have an `_21` row. If the numbering were unrelated, this would happen
by chance with the (hypergeometric) probability printed below. Metadata is shown only for cases where this check
passes *and* a single `_11` row exists. Missing fields are recorded per case.

In [ ]:
display(pd.Series(manifest["subject_mapping_check"]))
joins = pd.read_csv(f"{BUNDLE_DIR}/reports/metadata_join_report.csv")
print("verified joins:", int(joins["join_verified"].sum()), "/", len(joins))
missing = joins["missing_fields"].fillna("").str.split(";").explode()
print("missing-field counts:"); print(missing[missing != ""].value_counts())

### Audit check: unscaled features and fusion weights
The earlier code took the cosine of **raw** features. Here we recompute that legacy similarity on the real gallery
(adding back the raw mm³ volume it used) to show how much one feature dominated, and how far the fixed ranking differs.

In [ ]:
display(pd.Series(manifest["audit_unscaled_vs_fixed"]))
print("Fusion:", manifest["fusion"])
print("Sanity:", manifest["sanity"])

### Example rankings (held-out test queries)
No independent relevance judgments exist yet, so we **do not report Precision@K / mAP@K**. Instead we show example
rankings, and a *secondary* check against labels that were never used for ranking (MGMT, IDH1), compared with chance.

In [ ]:
from IPython.display import Image as IPImage, display as show
rankings = pd.read_csv(f"{BUNDLE_DIR}/reports/test_rankings.csv")
display(rankings.head(15))
print("Secondary label agreement (test queries, Top-%d):" % TOP_K)
display(pd.DataFrame(manifest["secondary_label_agreement_test"]).T)
q = rankings["query"].iloc[0]
print("query", q); show(IPImage(f"{BUNDLE_DIR}/previews/{q}.png"))
for m in rankings.loc[rankings["query"] == q, "match"].head(3):
    print("match", m); show(IPImage(f"{BUNDLE_DIR}/previews/{m}.png"))

## 7. Save the bundle to Drive
`gallery_bundle.zip` contains no raw scans. It is safe to download to your laptop and is a few MB.

In [ ]:
import shutil
zip_path = shutil.make_archive(f"{DRIVE_ROOT}/gallery_bundle", "zip", BUNDLE_DIR)
print(zip_path, f"{os.path.getsize(zip_path)/1e6:.1f} MB")
assert not list(pathlib.Path(BUNDLE_DIR).rglob("*.nii*")), "raw scans must not be in the bundle"

### Use it in Streamlit
The app does not read this bundle directly. It needs the 2D reference gallery that is built *from* it:
1. Unzip `gallery_bundle.zip` into `artifacts/gallery_bundle/` in the repository (the folder is git-ignored).
2. Run `python -m retrieval.gallery2d prepare` from the repository root. It reads the bundle's `gallery_metadata.csv`, `demo_metadata.csv` and `evaluation/secondary_labels.csv`, and writes the searchable gallery to `artifacts/gallery_2d/`.
3. Run `python -m streamlit run app.py`. The gallery location can be changed with the `GALLERY_2D_DIR` environment variable or Streamlit secret.

See the repository README ("Database configuration") for the full setup and an alternative that builds the bundle locally without Colab.

## 8. (Optional) Keep one held-out exam to test the upload path
This copies the five files of one **test** case into Drive so you can upload them in the app (about 11 MB).

In [ ]:
splits = pd.read_csv(f"{BUNDLE_DIR}/reports/splits.csv")
demo_subject = splits.loc[splits["split"] == "test", "case_id"].iloc[0]
exam_zip = f"{DRIVE_ROOT}/demo_exam_{demo_subject}.zip"
with zipfile.ZipFile(exam_zip, "w") as z:
    for f in sorted(pathlib.Path(RAW_DIR).glob(f"{demo_subject}_*.nii.gz")):
        z.write(f, f.name)
print("upload this zip in the app:", exam_zip)

## 9. Clean up
The raw cache on `/content` disappears with the runtime. If you set `KEEP_RAW_ON_DRIVE`, delete it here when done.

In [ ]:
if KEEP_RAW_ON_DRIVE and input("Delete raw cache on Drive? [y/N] ").lower() == "y":
    shutil.rmtree(RAW_DIR)

---
## Next (final milestone, needs a GPU runtime): team-trained segmentation
Use the **same split** (`reports/splits.csv`). Train nnU-Net v2 on gallery patients only, then evaluate Dice/HD95 on test patients:
```python
# from the repo: build a manifest with split=train for gallery rows, then
# python -m retrieval.cli prepare-nnunet --manifest <split_manifest.csv> --raw-root /content/nnunet_raw --dataset-id 501
# nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity
# nnUNetv2_train 501 3d_fullres 0
```
`dataset.json` now declares labels 1 necrotic core, 2 oedema, 3 enhancing tumour. Then compare retrieval with supplied vs model masks.